# Retrieval-based latent repair: CODI LLaMA-1B (ledger §126, Build 3)

**No training.** A bank of 2,048 training-pool rows is indexed by the value the model's own
output head reads at carrier thought 3. For every wrongly answered multi-step GSM8K
test question, a bank row holding the gold penultimate intermediate donates its carrier state
and/or its store K/V (position 4), and the answer is decoded again. Controls: a different-value
donor, the centroid of same-value rows, the whole donor tail, and correctly answered rows.
Predictions R1–R5 are in the ledger. No dataset to attach. Publish the output when finished.

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "3e2c73b4e95017156cad3919c1e5537b60635531"
REPO_DIR = "/kaggle/working/latent-reasoning"
CONFIG = "configs/official_codi_llama1b.yaml"
MODEL = "codi1b"
PRECISION = "float16"
OUTPUT_ROOT = "/kaggle/working/repair_codi1b"
CARRIER, STORE, EXTRA = "3", "4", []
V_SOURCE = "gold"
RUN_SMOKE = True

import glob, json, os, pathlib, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

### Environment and focused tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2", "accelerate==1.7.0",
                "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"],
    capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_official_codi_family.py", "tests/test_cache_carrier.py", "tests/test_latent_repair_runner.py", "tests/test_chain_pools.py"]], check=True)

def run_with_retry(command, attempts=3, wait=60):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(command)
        if result.returncode == 0:
            return
        if attempt == attempts:
            raise subprocess.CalledProcessError(result.returncode, command)
        print(f"runner exited with {result.returncode}; retrying in {wait}s ({attempt + 1}/{attempts})")
        time.sleep(wait)

## The repair run (smoke first, then full)

In [ ]:
def repair(out, smoke):
    cmd = [sys.executable, "-u", "scripts/run_codi_latent_repair.py", "--config", CONFIG, "--model-key", MODEL, "--output-dir", out,
           "--carrier", CARRIER, "--store", STORE, "--batch-size", "32", "--precision", PRECISION, "--device", "cuda", "--v-source", V_SOURCE]
    if EXTRA:
        cmd += ["--extra-carriers", *EXTRA]
    if smoke:
        cmd.append("--smoke")
    run_with_retry(cmd)
    return json.loads(open(pathlib.Path(out) / "summary.json").read())

if RUN_SMOKE:
    repair(OUTPUT_ROOT + "_smoke", True)
summary = repair(OUTPUT_ROOT, False)
import pandas as pd
from IPython.display import display
print(json.dumps(summary["report"]["targets"], indent=2)); print(json.dumps(summary["checks"], indent=2))
if "tool" in summary["report"]:
    print("tool:", json.dumps(summary["report"]["tool"], indent=2))
for part in ("primary", "with_donor", "carrier_already_right"):
    print("--- repair rate,", part)
    display(pd.DataFrame({k: {"rate": v["rate"], "ci95": v["ci95"], "n": v["n"]} for k, v in summary["report"]["repair"][part].items()}).T)
print("--- harm on correct controls"); display(pd.DataFrame({k: {"rate": v["rate"], "ci95": v["ci95"], "n": v["n"]} for k, v in summary["report"]["harm"].items()}).T)
print("Reminder: publish this notebook's output as a dataset (Output tab -> New Dataset).")